# 🎧 NexTune — Model Training Notebook
**Bluetooth Headphones Price Prediction | Amazon India**

> Colab: Runtime → Run all

In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib seaborn joblib
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import joblib, os, warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
sns.set_style('whitegrid')
print("✅ Libraries loaded")

## 1. Load & Prepare Data

In [ ]:
URL = 'https://raw.githubusercontent.com/ESpoorthy/NexTune/main/data/final-merged-dataset.csv' \
      if os.path.exists('/content') else '../data/final-merged-dataset.csv'
df = pd.read_csv(URL)

# Fix numeric types
num_cols = ['price_inr','rating','review_count','battery_life_hrs','bluetooth_version',
            'driver_size_mm','ipx_level','latency_ms','anc_db']
bin_cols = ['has_noise_cancellation','has_enc','has_usb_c','has_premium_codec',
            'has_touch_control','has_voice_assistant','has_fast_charge','has_dual_pairing',
            'has_gaming_mode','has_hi_res_audio','has_spatial_audio','has_low_latency']
for c in num_cols + bin_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')
for c in bin_cols:
    if c in df.columns:
        df[c] = df[c].fillna(0).astype(int)

# Drop rows with missing price
df = df.dropna(subset=['price_inr'])
df = df[df['price_inr'] > 0].reset_index(drop=True)
print(f"✅ Loaded: {len(df)} products")
print(f"   Price range: ₹{df.price_inr.min():.0f} – ₹{df.price_inr.max():.0f}")

## 2. Feature Engineering

In [ ]:
# Brand tier
brand_avg = df.groupby('brand')['price_inr'].mean()
df['brand_tier'] = df['brand'].apply(
    lambda b: 'premium' if brand_avg.get(b,0)>=10000 else ('mid' if brand_avg.get(b,0)>=3000 else 'budget')
)

# Encode categoricals
le_dict = {}
for col in ['category','brand','brand_tier','country_of_origin']:
    if col in df.columns:
        le = LabelEncoder()
        df[col+'_enc'] = le.fit_transform(df[col].fillna('Unknown').astype(str))
        le_dict[col] = le

# Engineered features
df['has_anc']        = df['has_noise_cancellation'].fillna(0).astype(int)
df['bt_major']       = df['bluetooth_version'].apply(lambda x: int(x) if pd.notna(x) else 5)
df['high_rating']    = (df['rating'] >= 4.0).astype(int)
df['price_per_hour'] = df['price_inr'] / (df['battery_life_hrs'].fillna(0) + 1)

print("✅ Feature engineering done")
print(f"   Brand tiers: {df.brand_tier.value_counts().to_dict()}")
print(f"   ANC products: {df.has_anc.sum()}")
print(f"   High-rated (≥4.0): {df.high_rating.sum()}")

## 3. Feature Selection — Keep Only Positive Correlations

Features with negative correlation to price are dropped. They add noise and hurt model accuracy.

In [ ]:
# Features with POSITIVE correlation to price (computed from EDA)
FEATURES = [
    'brand_tier_enc',    # corr=+0.776 ← strongest signal
    'price_per_hour',    # corr=+0.919 ← value metric
    'has_anc',           # corr=+0.392
    'rating',            # corr=+0.205
    'has_hi_res_audio',  # corr=+0.199
    'high_rating',       # corr=+0.174
    'has_spatial_audio', # corr=+0.165
    'bluetooth_version', # corr=+0.064
    'brand_enc',         # corr=+0.053
    'has_dual_pairing',  # corr=+0.052
    'has_premium_codec', # corr=+0.038
    'bt_major',          # corr=+0.036
    'anc_db',            # corr=+0.026
    'has_enc',           # corr=+0.026
    'review_count',      # corr=+0.025
    'ipx_level',         # corr=+0.016
    'driver_size_mm',    # corr=+0.016
    'has_low_latency',   # corr=+0.003
]

# DROPPED (negative correlation): latency_ms, battery_life_hrs, has_usb_c,
# has_gaming_mode, has_voice_assistant, has_ipx, has_touch_control,
# has_fast_charge, category_enc

FEATURES = [f for f in FEATURES if f in df.columns]
TARGET = 'price_inr'

X = df[FEATURES].fillna(df[FEATURES].median())
y = df[TARGET]

print(f"✅ Feature matrix: {X.shape}")
print(f"   {len(FEATURES)} features (all positive correlation with price)")
print(f"\nFeatures used:")
for f in FEATURES:
    corr_val = X[f].corr(y)
    print(f"   {f:25s}: r = {corr_val:+.3f}")

## 4. Train / Test Split & Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Train: {len(X_train)} samples  |  Test: {len(X_test)} samples")
print(f"Train price range: ₹{y_train.min():.0f} – ₹{y_train.max():.0f}")
print(f"Test  price range: ₹{y_test.min():.0f} – ₹{y_test.max():.0f}")

## 5. Train & Evaluate All Models

In [ ]:
models = {
    'Linear Regression':  LinearRegression(),
    'Ridge':              Ridge(alpha=10.0),
    'Lasso':              Lasso(alpha=10.0),
    'Random Forest':      RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1),
    'Gradient Boosting':  GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42),
}

results = []
for name, model in models.items():
    model.fit(X_train_sc, y_train)
    y_pred = model.predict(X_test_sc)
    r2   = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    mape = np.mean(np.abs((y_test - y_pred) / (y_test + 1))) * 100
    cv   = cross_val_score(model, X_train_sc, y_train, cv=5, scoring='r2').mean()
    results.append({'Model':name,'R²':r2,'CV R²':cv,'RMSE':rmse,'MAE':mae,'MAPE%':mape})
    print(f"{name:22s}  R²={r2:.3f}  CV R²={cv:.3f}  RMSE=₹{rmse:.0f}  MAE=₹{mae:.0f}")

results_df = pd.DataFrame(results).sort_values('R²', ascending=False).reset_index(drop=True)
print("\n--- Ranked Results ---")
print(results_df.round(3).to_string(index=False))

## 6. Model Comparison Charts

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

colors = ['#2ecc71' if r >= 0.70 else '#e74c3c' for r in results_df['R²']]
axes[0].barh(results_df['Model'], results_df['R²'], color=colors)
axes[0].axvline(0.70, color='black', ls='--', lw=1.5, label='Target R²=0.70')
axes[0].set_title('R² Score', fontweight='bold', fontsize=13)
axes[0].set_xlabel('R²'); axes[0].legend()
for i,v in enumerate(results_df['R²']): axes[0].text(v+0.005, i, f'{v:.3f}', va='center', fontweight='bold')

axes[1].barh(results_df['Model'], results_df['RMSE'], color='#378ADD')
axes[1].set_title('RMSE (₹) — lower is better', fontweight='bold', fontsize=13)
axes[1].set_xlabel('RMSE (INR)')
for i,v in enumerate(results_df['RMSE']): axes[1].text(v+5, i, f'₹{v:.0f}', va='center')

axes[2].barh(results_df['Model'], results_df['MAPE%'], color='#D85A30')
axes[2].set_title('MAPE% — lower is better', fontweight='bold', fontsize=13)
axes[2].set_xlabel('MAPE %')
for i,v in enumerate(results_df['MAPE%']): axes[2].text(v+0.2, i, f'{v:.1f}%', va='center')

plt.suptitle('Model Comparison — Positive-Correlation Features Only', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 7. Best Model — Actual vs Predicted

In [ ]:
best_name  = results_df.iloc[0]['Model']
best_model = models[best_name]
y_pred_best = best_model.predict(X_test_sc)

print(f"🏆 Best Model : {best_name}")
print(f"   R²         : {results_df.iloc[0]['R²']:.4f}")
print(f"   RMSE       : ₹{results_df.iloc[0]['RMSE']:.2f}")
print(f"   MAE        : ₹{results_df.iloc[0]['MAE']:.2f}")
print(f"   MAPE       : {results_df.iloc[0]['MAPE%']:.2f}%")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Actual vs Predicted scatter
lims = [min(y_test.min(), y_pred_best.min()), max(y_test.max(), y_pred_best.max())]
axes[0].scatter(y_test, y_pred_best, alpha=0.6, color='#378ADD', edgecolors='white', s=70)
axes[0].plot(lims, lims, 'r--', lw=2, label='Perfect prediction')
axes[0].set_xlabel('Actual Price (₹)', fontsize=12)
axes[0].set_ylabel('Predicted Price (₹)', fontsize=12)
axes[0].set_title(f'{best_name} — Actual vs Predicted', fontweight='bold', fontsize=13)
axes[0].legend()

# Residuals
residuals = y_test - y_pred_best
axes[1].scatter(y_pred_best, residuals, alpha=0.5, color='#1D9E75', s=60)
axes[1].axhline(0, color='red', ls='--', lw=2)
axes[1].set_xlabel('Predicted Price (₹)', fontsize=12)
axes[1].set_ylabel('Residual (Actual − Predicted)', fontsize=12)
axes[1].set_title('Residual Plot', fontweight='bold', fontsize=13)

plt.tight_layout(); plt.show()

## 8. Feature Importance

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    imp = best_model.feature_importances_
    label = 'Feature Importance'
else:
    imp = np.abs(best_model.coef_)
    label = '|Coefficient|'

feat_imp = pd.Series(imp, index=FEATURES).sort_values(ascending=True)

plt.figure(figsize=(10, 7))
colors = ['#2ecc71' if v >= feat_imp.median() else '#378ADD' for v in feat_imp.values]
feat_imp.plot(kind='barh', color=colors)
plt.title(f'Feature Importance — {best_name}', fontsize=14, fontweight='bold')
plt.xlabel(label)
plt.tight_layout(); plt.show()

print("Top 10 most important features:")
print(feat_imp.sort_values(ascending=False).head(10).round(4).to_string())

## 9. Save Model

In [ ]:
SAVE_DIR = '/content/models' if os.path.exists('/content') else '../models'
os.makedirs(SAVE_DIR, exist_ok=True)

joblib.dump(best_model, f'{SAVE_DIR}/price_predictor.pkl')
joblib.dump(scaler,     f'{SAVE_DIR}/scaler.pkl')
joblib.dump(le_dict,    f'{SAVE_DIR}/label_encoders.pkl')
joblib.dump(FEATURES,   f'{SAVE_DIR}/features.pkl')

print(f"✅ Saved to {SAVE_DIR}/")
print(f"   price_predictor.pkl")
print(f"   scaler.pkl")
print(f"   label_encoders.pkl")
print(f"   features.pkl")

if os.path.exists('/content'):
    from google.colab import files
    for f in ['price_predictor.pkl','scaler.pkl','label_encoders.pkl','features.pkl']:
        files.download(f'{SAVE_DIR}/{f}')

## 10. Predict Price for a New Headphone

Edit the specs below and run to get a recommended market price.

In [ ]:
def predict_price(brand, has_anc, is_tws, has_mic,
                  is_neckband=False, battery_hrs=30, driver_mm=10,
                  bt_version=5.3, rating=4.0, reviews=0,
                  has_hi_res=0, has_spatial=0, has_dual=0,
                  has_codec=0, has_enc=0, has_low_lat=0,
                  ipx_level=0, anc_db=0):

    category = 'true wireless earbuds' if is_tws else ('neckband' if is_neckband else 'over-ear headphone')
    tier = 'premium' if brand_avg.get(brand,0)>=10000 else ('mid' if brand_avg.get(brand,0)>=3000 else 'budget')

    def enc(col, val):
        le = le_dict.get(col)
        if le and val in le.classes_: return int(le.transform([val])[0])
        return 0

    row = {
        'brand_tier_enc':   enc('brand_tier', tier),
        'price_per_hour':   0,
        'has_anc':          int(has_anc),
        'rating':           rating,
        'has_hi_res_audio': int(has_hi_res),
        'high_rating':      int(rating >= 4.0),
        'has_spatial_audio':int(has_spatial),
        'bluetooth_version':bt_version,
        'brand_enc':        enc('brand', brand),
        'has_dual_pairing': int(has_dual),
        'has_premium_codec':int(has_codec),
        'bt_major':         int(bt_version),
        'anc_db':           anc_db,
        'has_enc':          int(has_enc),
        'review_count':     reviews,
        'ipx_level':        ipx_level,
        'driver_size_mm':   driver_mm,
        'has_low_latency':  int(has_low_lat),
    }

    input_vec = np.array([[row.get(f, 0) for f in FEATURES]])
    input_sc  = scaler.transform(input_vec)
    price     = best_model.predict(input_sc)[0]

    print("\n" + "=" * 50)
    print("       💡 PRICE RECOMMENDATION")
    print("=" * 50)
    print(f"  Brand          : {brand}  ({tier})")
    print(f"  Category       : {category}")
    print(f"  ANC            : {'Yes' if has_anc else 'No'}")
    print(f"  Bluetooth      : v{bt_version}")
    print(f"  Driver         : {driver_mm}mm")
    print(f"  Rating         : {rating}")
    print("-" * 50)
    print(f"  Predicted      : ₹{price:,.0f}")
    print(f"  Suggested range: ₹{price*0.9:,.0f} – ₹{price*1.1:,.0f}  (±10%)")
    print("=" * 50)

# ✏️ Edit these specs for your new headphone
predict_price(
    brand='Boat', has_anc=True, is_tws=True, has_mic=True,
    battery_hrs=40, driver_mm=10, bt_version=5.3,
    rating=4.2, reviews=0,
    has_hi_res=0, has_spatial=0, has_dual=0,
    has_codec=0, has_enc=1, has_low_lat=1,
    ipx_level=4, anc_db=30
)